In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# --- Source SQL DB connection (used for row-count validation) ---
jdbc_hostname = "fintec-db-server.database.windows.net"
jdbc_database = "fintec-db"
jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:1433;database={jdbc_database}"

jdbc_username = dbutils.secrets.get(scope="fintech-kv-scope", key="sql-username")
jdbc_password = dbutils.secrets.get(scope="fintech-kv-scope", key="sql-password")

connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# --- Paths ---
base_path = "abfss://fintech@fintechstgadlsaccdev.dfs.core.windows.net/bronze/"
output_base_path = "abfss://fintech@fintechstgadlsaccdev.dfs.core.windows.net/silver/"


def validate_row_counts(bronze_df, table_name):
    """Compare source SQL count vs Bronze count. Fail fast on mismatch."""
    source_count = spark.read.jdbc(
        url=jdbc_url, table=f"fintech.{table_name}", properties=connection_properties
    ).count()
    bronze_count = bronze_df.count()

    print(f"{table_name} - Source count: {source_count}, Bronze count: {bronze_count}")

    if source_count != bronze_count:
        raise Exception(f"Row count mismatch for '{table_name}': source={source_count}, bronze={bronze_count}")

    print(f"[VALIDATED] {table_name}: counts match ({bronze_count} rows)")

from delta.tables import DeltaTable

def write_to_silver(df, table_name, primary_key, output_base_path):
    """
    Upserts into Silver using MERGE instead of blind overwrite.
    Makes writes idempotent -- safe to re-run without duplicating data,
    and updates existing records in place (matches real SCD Type 1 pattern).
    """
    full_path = f"{output_base_path}{table_name}/"

    if DeltaTable.isDeltaTable(spark, full_path):
        delta_table = DeltaTable.forPath(spark, full_path)
        (delta_table.alias("target")
            .merge(df.alias("source"), f"target.{primary_key} = source.{primary_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"{table_name}: merged (upsert)")
    else:
        df.write.format("delta").mode("overwrite").save(full_path)
        print(f"{table_name}: created (initial load)")

def validate_data_quality(df, table_name):
    total_records = df.count()
    null_records = df.filter(col("CustomerID").isNull()).count() if "CustomerID" in df.columns else 0
    quality_score = ((total_records - null_records) / total_records * 100) if total_records > 0 else 0
    print(f"{table_name} - Total records: {total_records}, Quality score: {quality_score:.2f}%")
    if quality_score < 95:
        print(f"WARNING: Data quality below threshold for {table_name}")
    return quality_score


def clean_and_validate_customers(df):
    print("Processing customers data")
    validate_data_quality(df, "customers")

    cleaned_df = df.withColumn("FirstName", initcap(trim(col("FirstName")))) \
                 .withColumn("LastName", initcap(trim(col("LastName")))) \
                 .withColumn("Email", lower(trim(col("Email")))) \
                 .withColumn("City", initcap(trim(col("City")))) \
                 .withColumn("State", upper(trim(col("State")))) \
                 .withColumn("Country", initcap(trim(col("Country"))))

    enriched_df = cleaned_df.withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName"))) \
                           .withColumn("CustomerAge", round(datediff(current_date(), col("SignupDate")) / 365.25, 0)) \
                           .withColumn("CustomerSegment",
                                       when(col("SignupDate") >= "2024-01-01", "New")
                                       .when(col("SignupDate") >= "2023-01-01", "Recent")
                                       .otherwise("Established")) \
                           .withColumn("MaskedEmail", concat(lit("***@"), substring_index(col("Email"), "@", -1))) \
                           .withColumn("CustomerTier",
                                       when(col("SignupDate") >= "2024-01-01", "Premium")
                                       .when(col("SignupDate") >= "2023-06-01", "Gold")
                                       .otherwise("Standard"))

    invalid_emails = enriched_df.filter(~col("Email").rlike(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')).count()
    if invalid_emails > 0:
        print(f"WARNING: Found {invalid_emails} invalid email addresses")

    return enriched_df


def clean_and_validate_accounts(df):
    print("Processing accounts data")
    validate_data_quality(df, "accounts")

    cleaned_df = df.withColumn("AccountType",
                              when(col("AccountType").isin(["Savings", "Saving"]), "Savings")
                              .when(col("AccountType").isin(["Checking", "Check"]), "Checking")
                              .when(col("AccountType").isin(["Investment", "Invest"]), "Investment")
                              .when(col("AccountType").isin(["Business", "Biz"]), "Business")
                              .otherwise(col("AccountType"))) \
                  .withColumn("Balance", when(col("Balance") < 0, 0.0).otherwise(col("Balance")))

    enriched_df = cleaned_df.withColumn("AccountAgeYears", round(datediff(current_date(), col("OpenDate")) / 365.25, 2)) \
                           .withColumn("AccountStatus",
                                       when(col("Balance") == 0, "Inactive")
                                       .when(col("Balance") < 1000, "Low Balance")
                                       .when(col("Balance") < 10000, "Standard")
                                       .otherwise("Premium")) \
                           .withColumn("AccountTier",
                                       when(col("Balance") >= 50000, "Gold")
                                       .when(col("Balance") >= 25000, "Silver")
                                       .when(col("Balance") >= 10000, "Bronze")
                                       .otherwise("Basic")) \
                           .withColumn("MonthlyInterest", (col("Balance") * 0.02 / 12).cast(DecimalType(18, 2))) \
                           .withColumn("IsHighValue", col("Balance") >= 50000)

    negative_balances = enriched_df.filter(col("Balance") < 0).count()
    if negative_balances > 0:
        print(f"WARNING: Found {negative_balances} accounts with negative balances")

    return enriched_df


def clean_and_validate_loans(df):
    print("Processing loans data")
    validate_data_quality(df, "loans")

    cleaned_df = df.withColumn("LoanType",
                              when(col("LoanType").isin(["Personal", "Personal Loan"]), "Personal")
                              .when(col("LoanType").isin(["Home", "Home Loan", "Mortgage"]), "Home")
                              .when(col("LoanType").isin(["Car", "Auto", "Car Loan"]), "Car")
                              .when(col("LoanType").isin(["Education", "Student", "Education Loan"]), "Education")
                              .when(col("LoanType").isin(["Business", "Business Loan"]), "Business")
                              .otherwise(col("LoanType"))) \
                  .withColumn("LoanAmount", when(col("LoanAmount") <= 0, 1000.0).otherwise(col("LoanAmount"))) \
                  .withColumn("InterestRate",
                              when(col("InterestRate") < 0, 0.0)
                              .when(col("InterestRate") > 30.0, 30.0)
                              .otherwise(col("InterestRate")))

    enriched_df = cleaned_df.withColumn("LoanDurationYears", round(datediff(col("LoanEndDate"), col("LoanStartDate")) / 365.25, 2)) \
                           .withColumn("TotalInterest", (col("LoanAmount") * col("InterestRate") / 100).cast(DecimalType(18, 2))) \
                           .withColumn("MonthlyPayment", ((col("LoanAmount") + col("TotalInterest")) / (col("LoanDurationYears") * 12)).cast(DecimalType(18, 2))) \
                           .withColumn("LoanStatus",
                                       when(col("LoanEndDate") < current_date(), "Completed")
                                       .when(col("LoanStartDate") > current_date(), "Pending")
                                       .otherwise("Active")) \
                           .withColumn("RiskCategory",
                                       when((col("LoanAmount") > 50000) & (col("InterestRate") > 10), "High Risk")
                                       .when((col("LoanAmount") > 25000) & (col("InterestRate") > 7), "Medium Risk")
                                       .otherwise("Low Risk")) \
                           .withColumn("LoanToValueRatio", (col("LoanAmount") / 100000).cast(DecimalType(5, 2))) \
                           .withColumn("IsHighRisk", (col("LoanAmount") > 50000) & (col("InterestRate") > 10))

    invalid_dates = enriched_df.filter(col("LoanStartDate") >= col("LoanEndDate")).count()
    if invalid_dates > 0:
        print(f"WARNING: Found {invalid_dates} loans with invalid date ranges")

    return enriched_df


def clean_and_validate_transactions(df):
    print("Processing transactions data")
    validate_data_quality(df, "transactions")

    cleaned_df = df.withColumn("TransactionType",
                              when(col("TransactionType").isin(["Deposit", "Credit"]), "Deposit")
                              .when(col("TransactionType").isin(["Withdrawal", "Debit"]), "Withdrawal")
                              .when(col("TransactionType").isin(["Transfer", "Transfer In", "Transfer Out"]), "Transfer")
                              .when(col("TransactionType").isin(["Fee", "Service Fee", "Maintenance Fee"]), "Fee")
                              .otherwise(col("TransactionType"))) \
                  .withColumn("Amount", when(col("Amount") <= 0, 0.01).otherwise(col("Amount"))) \
                  .withColumn("Description", trim(col("Description")))

    enriched_df = cleaned_df.withColumn("TransactionCategory",
                                       when(col("TransactionType") == "Deposit", "Income")
                                       .when(col("TransactionType") == "Withdrawal", "Expense")
                                       .when(col("TransactionType") == "Transfer", "Transfer")
                                       .when(col("TransactionType") == "Fee", "Fee")
                                       .otherwise("Other")) \
                           .withColumn("TransactionSize",
                                       when(col("Amount") >= 10000, "Large")
                                       .when(col("Amount") >= 1000, "Medium")
                                       .when(col("Amount") >= 100, "Small")
                                       .otherwise("Micro")) \
                           .withColumn("TransactionDayOfWeek", date_format(col("TransactionDate"), "EEEE")) \
                           .withColumn("TransactionMonth", date_format(col("TransactionDate"), "MM")) \
                           .withColumn("TransactionYear", date_format(col("TransactionDate"), "yyyy")) \
                           .withColumn("IsWeekend",
                                       when(col("TransactionDayOfWeek").isin(["Saturday", "Sunday"]), True)
                                       .otherwise(False)) \
                           .withColumn("IsLargeTransaction", col("Amount") >= 10000)

    return enriched_df


def clean_and_validate_payments(df):
    print("Processing payments data")
    validate_data_quality(df, "payments")

    cleaned_df = df.withColumn("PaymentMethod",
                              when(col("PaymentMethod").isin(["Credit Card", "Credit"]), "Credit Card")
                              .when(col("PaymentMethod").isin(["Debit Card", "Debit"]), "Debit Card")
                              .when(col("PaymentMethod").isin(["Cash", "Cash Payment"]), "Cash")
                              .when(col("PaymentMethod").isin(["Bank Transfer", "Wire Transfer", "ACH"]), "Bank Transfer")
                              .when(col("PaymentMethod").isin(["Check", "Cheque"]), "Check")
                              .otherwise(col("PaymentMethod"))) \
                  .withColumn("PaymentAmount", when(col("PaymentAmount") <= 0, 0.01).otherwise(col("PaymentAmount")))

    enriched_df = cleaned_df.withColumn("DaysSinceLastPayment", datediff(current_date(), col("PaymentDate"))) \
                           .withColumn("PaymentSize",
                                       when(col("PaymentAmount") >= 5000, "Large")
                                       .when(col("PaymentAmount") >= 1000, "Medium")
                                       .when(col("PaymentAmount") >= 100, "Small")
                                       .otherwise("Micro")) \
                           .withColumn("PaymentMethodCategory",
                                       when(col("PaymentMethod").isin(["Credit Card", "Debit Card"]), "Card")
                                       .when(col("PaymentMethod") == "Cash", "Cash")
                                       .when(col("PaymentMethod") == "Bank Transfer", "Electronic")
                                       .when(col("PaymentMethod") == "Check", "Check")
                                       .otherwise("Other")) \
                           .withColumn("IsLatePayment", col("DaysSinceLastPayment") > 30) \
                           .withColumn("IsLargePayment", col("PaymentAmount") >= 5000)

    return enriched_df


# --- Process each table: validate counts, then clean, then write ---
try:
    customers_df = spark.read.parquet(f"{base_path}Customers/Customers.parquet")
    validate_row_counts(customers_df, "customers")
    customers_processed = clean_and_validate_customers(customers_df)
    write_to_silver(customers_processed, "Customers", "CustomerID", output_base_path)
    print("Customers processing completed")

    accounts_df = spark.read.parquet(f"{base_path}Accounts/Accounts.parquet")
    validate_row_counts(accounts_df, "accounts")
    accounts_processed = clean_and_validate_accounts(accounts_df)
    write_to_silver(accounts_processed, "Accounts", "AccountID", output_base_path)
    print("Accounts processing completed")

    loans_df = spark.read.parquet(f"{base_path}Loans/Loans.parquet")
    validate_row_counts(loans_df, "loans")
    loans_processed = clean_and_validate_loans(loans_df)
    write_to_silver(loans_processed, "Loans", "LoanID", output_base_path)
    print("Loans processing completed")

    transactions_df = spark.read.parquet(f"{base_path}Transactions/Transactions.parquet")
    validate_row_counts(transactions_df, "transactions")
    transactions_processed = clean_and_validate_transactions(transactions_df)
    write_to_silver(transactions_processed, "Transactions", "TransactionID", output_base_path)
    print("Transactions processing completed")

    payments_df = spark.read.parquet(f"{base_path}Payments/Payments.parquet")
    validate_row_counts(payments_df, "payments")
    payments_processed = clean_and_validate_payments(payments_df)
    write_to_silver(payments_processed, "Payments", "PaymentID", output_base_path)
    print("Payments processing completed")

    print("Bronze To Silver Processing Completed Successfully!")

except Exception as e:
    print(f"Error: {str(e)}")
    raise

customers - Source count: 30, Bronze count: 30
[VALIDATED] customers: counts match (30 rows)
Processing customers data
customers - Total records: 30, Quality score: 100.00%
Customers: merged (upsert)
Customers processing completed
accounts - Source count: 30, Bronze count: 30
[VALIDATED] accounts: counts match (30 rows)
Processing accounts data
accounts - Total records: 30, Quality score: 100.00%
Accounts: merged (upsert)
Accounts processing completed
loans - Source count: 300, Bronze count: 300
[VALIDATED] loans: counts match (300 rows)
Processing loans data
loans - Total records: 300, Quality score: 100.00%
Loans: merged (upsert)
Loans processing completed
transactions - Source count: 200, Bronze count: 200
[VALIDATED] transactions: counts match (200 rows)
Processing transactions data
transactions - Total records: 200, Quality score: 100.00%
Transactions: merged (upsert)
Transactions processing completed
payments - Source count: 75, Bronze count: 75
[VALIDATED] payments: counts match